# 🔴 EducaStock — Classificação de Risco de Vencimento (Random Forest na nuvem)

> **ML Caso de Uso #1** — ONG Casa da Criança  
> Modelo: **Random Forest** treinado e executado direto no Colab.  
> Saída gravada em `risk_classifications` no Firestore — o app apenas LÊ.

## Fluxo
1. Autentica com Firebase Admin SDK
2. Exporta lotes (`batches`) ativos do Firestore
3. Extrai as 4 features normalizadas por lote
4. Aumenta o dataset com amostras sintéticas (se necessário) e treina o Random Forest
5. Roda `predict_proba` em **todos os lotes ativos**
6. Grava `level` + `probabilities` em `risk_classifications` (1 doc por lote)

## Por que NÃO usamos mais TFLite
- App fica mais leve (sem dependência nativa pesada)
- Mesmo padrão do ML #2 (Prophet → Firestore → app)
- Pra atualizar o modelo, basta rodar esse notebook de novo — ninguém precisa atualizar o app
- Roda igual em Android, iOS e Web

## Schema gravado em `risk_classifications` (1 doc por lote, doc.id = batchId)
```json
{
  "batchId":       "lote_abc123",
  "productName":   "Arroz tipo 1 - 5kg",
  "level":         "vermelho",
  "probabilities": [0.05, 0.18, 0.77],
  "source":        "random_forest",
  "modelVersion":  "rf_v1",
  "generatedAt":   "2026-05-25T00:00:00Z"
}
```

## Pré-requisitos
- Arquivo `serviceAccountKey.json` do Firebase Admin (upload na próxima célula)
- Para gerar: Firebase Console → Configurações → Contas de serviço → Gerar nova chave privada

In [ ]:
# ─── Instalar dependências ─────────────────────────────────────────────────
!pip install firebase-admin pandas numpy scikit-learn matplotlib seaborn --quiet
print("✅ Dependências instaladas")

In [ ]:
# ─── Upload do arquivo de credenciais Firebase ─────────────────────────────
from google.colab import files

print("Faça upload do serviceAccountKey.json")
print("Caminho: Firebase Console → Configurações → Contas de Serviço → Gerar nova chave privada")
uploaded = files.upload()
SERVICE_ACCOUNT_KEY = list(uploaded.keys())[0]
print(f"\n✅ Arquivo carregado: {SERVICE_ACCOUNT_KEY}")

In [ ]:
# ─── Inicializar Firebase Admin ────────────────────────────────────────────
import firebase_admin
from firebase_admin import credentials, firestore

if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_KEY)
    firebase_admin.initialize_app(cred)

db = firestore.client()
print("✅ Firebase Admin inicializado")

In [ ]:
# ─── Exportar lotes ATIVOS do Firestore ────────────────────────────────────
import pandas as pd
from datetime import datetime, timezone

ACTIVE_STATUSES = {'disponivel', 'reservado'}   # estados ativos no Firestore

def export_active_batches():
    """Exporta os lotes ATIVOS do Firestore como DataFrame."""
    docs = db.collection('batches').stream()
    rows = []
    for doc in docs:
        d = doc.to_dict()
        if (d.get('status') or 'disponivel') not in ACTIVE_STATUSES:
            continue
        d['id'] = doc.id
        rows.append(d)
    return pd.DataFrame(rows)

df_batches = export_active_batches()
print(f"📦 Lotes ativos encontrados: {len(df_batches)}")
if len(df_batches) > 0:
    cols_pref = ['id','productName','quantity','initialQuantity','expiryDate','noExpiry','status']
    cols_show = [c for c in cols_pref if c in df_batches.columns]
    print("\nPrimeiros lotes:")
    print(df_batches[cols_show].head())

In [ ]:
# ─── Extração de features (mesma lógica do app: BatchFeatures.fromBatch) ───
import numpy as np

MAX_DAYS = 365.0

def parse_date(val):
    """Converte string ISO ou Timestamp do Firestore para datetime UTC."""
    if val is None:
        return None
    if hasattr(val, 'timestamp') and callable(val.timestamp):
        try:
            return datetime.fromtimestamp(val.timestamp(), tz=timezone.utc)
        except Exception:
            pass
    if hasattr(val, 'seconds'):  # Firestore Timestamp
        return datetime.fromtimestamp(val.seconds, tz=timezone.utc)
    if isinstance(val, datetime):
        return val if val.tzinfo else val.replace(tzinfo=timezone.utc)
    if isinstance(val, str):
        try:
            return datetime.fromisoformat(val.replace('Z', '+00:00'))
        except ValueError:
            return None
    return None

def extract_features_row(row, now):
    """Retorna (features_4d, days_to_expiry, no_expiry) para um lote."""
    no_expiry = bool(row.get('noExpiry', False))
    expiry = parse_date(row.get('expiryDate'))
    entry  = parse_date(row.get('entryDate'))
    qty    = float(row.get('quantity', 0) or 0)
    init   = float(row.get('initialQuantity', qty) or qty)
    if init <= 0:
        init = max(qty, 1)

    if no_expiry or expiry is None:
        days_to_expiry = int(MAX_DAYS)
        days_to_expiry_norm = 1.0
    else:
        days_to_expiry = (expiry - now).days
        days_to_expiry_norm = float(np.clip(days_to_expiry / MAX_DAYS, 0.0, 1.0))

    quantity_ratio = float(np.clip(qty / init, 0.0, 1.0))

    if entry is None:
        days_since_entry_norm = 0.0
    else:
        days_since_entry = (now - entry).days
        days_since_entry_norm = float(np.clip(days_since_entry / MAX_DAYS, 0.0, 1.0))

    is_no_expiry = 1.0 if no_expiry else 0.0
    feats = [days_to_expiry_norm, quantity_ratio, days_since_entry_norm, is_no_expiry]
    return feats, days_to_expiry, no_expiry

def label_from_rules(days_to_expiry, quantity_ratio, days_since_entry_norm, no_expiry):
    """Mesma lógica do RuleBasedRiskClassifier no app Flutter.
    0 = verde, 1 = amarelo, 2 = vermelho."""
    if no_expiry:
        return 0
    if days_to_expiry <= 0:
        return 2
    if days_to_expiry <= 7:
        return 2
    if days_to_expiry <= 30:
        if quantity_ratio > 0.8 and days_to_expiry <= 20:
            return 2
        return 1
    stale = days_since_entry_norm > (60 / MAX_DAYS) and quantity_ratio > 0.8
    if stale and days_to_expiry <= 90:
        return 1
    return 0

now = datetime.now(timezone.utc)
if len(df_batches) > 0:
    X_real, y_real = [], []
    for _, row in df_batches.iterrows():
        feats, dte, ne = extract_features_row(row, now)
        label = label_from_rules(dte, feats[1], feats[2], ne)
        X_real.append(feats); y_real.append(label)
    X_real = np.array(X_real, dtype=np.float32)
    y_real = np.array(y_real, dtype=np.int32)
    print(f"✅ Features extraídas de {len(X_real)} lotes reais")
    unique, counts = np.unique(y_real, return_counts=True)
    print(f"   Distribuição: {dict(zip(unique.tolist(), counts.tolist()))}  (0=verde, 1=amarelo, 2=vermelho)")
else:
    X_real = np.empty((0, 4), dtype=np.float32)
    y_real = np.empty(0, dtype=np.int32)
    print("⚠️  Nenhum lote real. Treinaremos apenas com sintético.")

In [ ]:
# ─── Gerar dataset sintético pra robustez (complementa os reais) ───────────
RANDOM_SEED = 42
TARGET_SAMPLES = max(4000, len(X_real) * 10)

def generate_synthetic(n_samples: int, seed: int = 42):
    rng = np.random.default_rng(seed)
    X, y = [], []
    for _ in range(n_samples):
        is_no_expiry = bool(rng.choice([False, True], p=[0.65, 0.35]))
        if is_no_expiry:
            days_to_expiry = int(MAX_DAYS)
            dnorm = 1.0
        else:
            days_to_expiry = int(rng.integers(0, int(MAX_DAYS) + 1))
            dnorm = float(days_to_expiry) / MAX_DAYS
        qr = float(rng.uniform(0.05, 1.0))
        ds = float(rng.uniform(0.0, 1.0))
        label = label_from_rules(days_to_expiry, qr, ds, is_no_expiry)
        # 5% de ruído pra melhor generalização
        if rng.random() < 0.05:
            label = int(rng.integers(0, 3))
        X.append([dnorm, qr, ds, 1.0 if is_no_expiry else 0.0])
        y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)

n_synth = max(TARGET_SAMPLES - len(X_real), 500)
X_synth, y_synth = generate_synthetic(n_synth, seed=RANDOM_SEED)

if len(X_real) > 0:
    X_all = np.vstack([X_real, X_synth])
    y_all = np.concatenate([y_real, y_synth])
else:
    X_all, y_all = X_synth, y_synth

print(f"📊 Dataset final: {len(X_all)} amostras  (reais: {len(X_real)} | sintéticos: {len(X_synth)})")
labels_map = {0: 'verde', 1: 'amarelo', 2: 'vermelho'}
for u, c in zip(*np.unique(y_all, return_counts=True)):
    print(f"   {labels_map[int(u)]}: {c} ({100*c/len(y_all):.1f}%)")

In [ ]:
# ─── Treinar Random Forest ─────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all
)

clf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    class_weight='balanced',
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = float((y_pred == y_test).mean())
print(f"\n=== Random Forest — Acurácia: {acc:.4f} ===")
print(classification_report(y_test, y_pred, target_names=['verde', 'amarelo', 'vermelho']))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['verde','amarelo','vermelho'],
            yticklabels=['verde','amarelo','vermelho'], ax=ax)
ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
ax.set_title('Matriz de Confusão — Classificação de Risco')
plt.tight_layout(); plt.show()

fig2, ax2 = plt.subplots(figsize=(6, 3))
feature_names = ['days_to_expiry_norm','quantity_ratio','days_since_entry_norm','is_no_expiry']
ax2.barh(feature_names, clf.feature_importances_, color='steelblue')
ax2.set_xlabel('Importância'); ax2.set_title('Importância das Features')
plt.tight_layout(); plt.show()

In [ ]:
# ─── Classificar TODOS os lotes ativos ─────────────────────────────────────
MODEL_VERSION = 'rf_v1'
LABELS = ['verde', 'amarelo', 'vermelho']

classifications = []
if len(df_batches) > 0:
    feats_matrix = np.array([extract_features_row(r, now)[0] for _, r in df_batches.iterrows()], dtype=np.float32)
    proba_matrix = clf.predict_proba(feats_matrix)
    predicted    = np.argmax(proba_matrix, axis=1)

    for (_, row), probs, pred in zip(df_batches.iterrows(), proba_matrix, predicted):
        classifications.append({
            'batchId':       row['id'],
            'productId':     row.get('productId', ''),
            'productName':   row.get('productName', 'Produto'),
            'level':         LABELS[int(pred)],
            'probabilities': [float(p) for p in probs],  # [verde, amarelo, vermelho]
            'source':        'random_forest',
            'modelVersion':  MODEL_VERSION,
            'generatedAt':   datetime.now(timezone.utc).isoformat(),
        })

    df_res = pd.DataFrame(classifications)
    print(f"✅ {len(classifications)} lotes classificados\n")
    print(df_res.groupby('level').size().rename('quantidade').to_string())
else:
    print("Nenhum lote ativo para classificar.")

In [ ]:
# ─── Gravar resultado em risk_classifications no Firestore ─────────────────
def write_classifications_to_firestore(rows: list, batch_size: int = 400) -> int:
    col = db.collection('risk_classifications')
    written = 0
    batch = db.batch()
    for i, item in enumerate(rows, start=1):
        ref = col.document(item['batchId'])  # doc.id == batchId pra fácil lookup
        batch.set(ref, item, merge=False)
        if i % batch_size == 0:
            batch.commit()
            batch = db.batch()
            written = i
            print(f"   Batch enviado: {written} registros…")
    if rows and len(rows) % batch_size != 0:
        batch.commit()
    return len(rows)

if classifications:
    n = write_classifications_to_firestore(classifications)
    print(f"✅ {n} classificações gravadas em risk_classifications/")
else:
    print("⚠️  Nada para gravar.")

In [ ]:
# ─── Resumo final ──────────────────────────────────────────────────────────
import json

summary = {
    'model_version':       MODEL_VERSION,
    'trained_at':          datetime.now(timezone.utc).isoformat(),
    'real_batches':        int(len(X_real)),
    'synthetic_samples':   int(len(X_synth)),
    'total_train_samples': int(len(X_all)),
    'rf_accuracy_test':    float(acc),
    'classified_batches':  int(len(classifications)),
    'features': ['days_to_expiry_norm','quantity_ratio','days_since_entry_norm','is_no_expiry'],
    'labels':   ['verde','amarelo','vermelho'],
    'output_collection': 'risk_classifications',
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\n📋 PRÓXIMOS PASSOS:")
print("   • Abra o app — o widget de risco no Dashboard já vai mostrar os novos níveis em tempo real.")
print("   • Para atualizar de novo: re-execute este notebook (recomendado 1× por semana).")
print("   • O app usa fallback automático (regras determinísticas) para lotes sem classificação.")

## 📌 Operação

### Quando rodar
Execute este notebook **1× por semana** ou após grandes movimentações de estoque (entrada/saída em massa).

### Coleção gerada: `risk_classifications`
- **Document ID**: `batchId` (para lookup direto pelo lote)
- **Schema**: ver topo do notebook
- **Quem lê**: app Flutter via `RiskClassificationFirestoreRepository.watchAll()` (Stream em tempo real)

### Plano B no app
Se um lote ainda não tem classificação na coleção (recém-cadastrado, notebook nunca rodou), o app usa
o `RuleBasedRiskClassifier` local automaticamente — então o app **nunca exibe nada vazio**.

### Boas práticas
- Re-executar após grandes mudanças no estoque
- Versionar o modelo com `modelVersion` (`rf_v1`, `rf_v2`…) ao mudar hiperparâmetros
- Acompanhar a acurácia no conjunto de teste e a matriz de confusão a cada execução